# omnisus no Colab: do DATASUS a uma tabela citável

Este notebook baixa os óbitos de Roraima em 2023 (SIM), põe rótulos nos códigos, confere
as colunas e gera a citação dos arquivos usados. Rode as células em ordem
(**Ambiente de execução → Executar tudo** também funciona).

> **Trabalho em construção.** Confira as contagens com o DATASUS e os rótulos com a
> fonte antes de publicar: [como conferir](https://raphaelfh.github.io/omnisus/dicionario/).

## 1. Instalar

Instala a versão fixada abaixo. Leva cerca de um minuto.

In [ ]:
%pip install -q "omnisus @ git+https://github.com/raphaelfh/omnisus@v0.1.0"

## 2. Onde guardar o lake (opcional)

Sem esta célula, os dados ficam em `/content/data/raw` e somem quando o ambiente do Colab
é desligado. Com ela, ficam no seu Google Drive e um novo `load` do mesmo recorte não
baixa nada de novo.

In [ ]:
from google.colab import drive

import omnisus as odb

drive.mount("/content/drive")
odb.set_lake_dir("/content/drive/MyDrive/omnisus")

## 3. O que o DATASUS publica

`available` consulta o servidor e lista os recortes que existem. A página
[Bases e argumentos](https://raphaelfh.github.io/omnisus/datasets/) diz o que cada base
aceita em `years`, `ufs` e `months`.

In [ ]:
import omnisus as odb

odb.available("sim_obitos", years=[2023], ufs=["RR"])

## 4. Baixar e ler

`load` importa o recorte para o lake e devolve as linhas com os códigos como o DATASUS
publicou.

In [ ]:
dados = odb.load("sim_obitos", years=[2023], ufs=["RR"])
dados.height

## 5. Rótulos e conferência

`label` acrescenta `<coluna>_rotulo` a partir do dicionário; um código que o dicionário
não conhece fica sem rótulo. `check_columns` mostra vazios, códigos sem rótulo e datas
fora do esperado em cada coluna.

In [ ]:
dados = odb.label("sim_obitos", dados, columns=["sexo", "racacor"])
dados.group_by("sexo_rotulo").len().sort("len", descending=True)

In [ ]:
odb.check_columns("sim_obitos", dados)

## 6. Citar

A citação nomeia os arquivos do servidor, o SHA-256 de cada um e o snapshot do lake.
Guarde-a junto com o resultado.

In [ ]:
with odb.LakeReader() as lake:
    print(odb.cite(lake, dataset="sim_obitos").text)

## Próximos passos

- Troque `"sim_obitos"`, `years` e `ufs` pela base e o recorte da sua pergunta.
- Se usou o Drive, rode `drive.flush_and_unmount()` antes de fechar, para todos os
  arquivos chegarem lá.
- Os [notebooks de cada base](https://github.com/raphaelfh/omnisus/tree/main/notebooks)
  seguem seis etapas, do que a base registra até a citação.